# Cultural-Track Full Evaluation with Qwen/Qwen2.5-VL-72B-Instruct

This notebook is the **cultural companion** to the game-track EDA report. It:

1. Loads **all 14** cultural cases (`data/Test_Cases/04_cultural`, C1-C14) and the
   location-stripped bank `data/Test_Cases/cultural_without_loc/`.
2. Prompts **`Qwen/Qwen2.5-VL-72B-Instruct`** to produce a declarative world model
   (players / states / actions / transition probabilities) for every case.
3. Runs the repository's **full evaluation** on each generated model: Lane 0
   (strict JSON parse), Lane 1 (mechanical v2 checks), structural EDA
   (`WorldModelEDA`), and Lane 3 (expect/avoid checklist) via
   `evaluate_cultural.evaluate_candidate`.
4. Renders the **cross-case EDA heatmaps** and compares the `04_cultural` bank
   against `cultural_without_loc` to confirm the **same level of expected output**
   is reached when the place name (Japan, Seoul, Korea, ...) is omitted.

> Credentials: set `QWEN_BASE_URL` and `QWEN_API_KEY` in the environment (or a
> `.env` in the repo root) to point at an OpenAI-compatible server serving the
> model, e.g. vLLM / SGLang / Together. If neither is set, the client falls back
> to local Ollama (model tag `qwen2.5-vl:72b`).

In [ ]:
import json, os, sys, time
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
for _lvl in range(4):
    if (ROOT / "evaluate_cultural.py").exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from evaluate_cultural import load_cases, evaluate_candidate

ORIG_DIR = ROOT / "data" / "Test_Cases" / "04_cultural"
WL_DIR   = ROOT / "data" / "Test_Cases" / "cultural_without_loc"
OUT_RUN  = ROOT / "experiments" / "cultural_qwen"
OUT_RUN.mkdir(parents=True, exist_ok=True)
print("root:", ROOT)

## 1. Load the case banks

In [ ]:
bank_with_loc = {c.cid: c for c in load_cases(cases_dir=ORIG_DIR)}
bank_without  = {c.cid: c for c in load_cases(cases_dir=WL_DIR)}
ORDER = [f"C{n}" for n in range(1, 15)]
assert len(bank_with_loc) == 14 and len(bank_without) == 14, \
    (len(bank_with_loc), len(bank_without))
print("with-loc:", ", ".join(bank_with_loc))
print("without-loc:", ", ".join(bank_without))

## 2. OpenAI-compatible client for Qwen/Qwen2.5-VL-72B-Instruct

Use the **OpenAI SDK** against any compatible server. If no endpoint is
configured, fall back to local **Ollama** (`ollama pull qwen2.5-vl:72b`).

In [ ]:
import shutil
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

MODEL = "Qwen/Qwen2.5-VL-72B-Instruct"
OLLAMA_MODEL = os.environ.get("QWEN_OLLAMA_MODEL", "qwen2.5-vl:72b")

def build_client():
    base_url = os.environ.get("QWEN_BASE_URL") or os.environ.get("LLM_BASE_URL")
    api_key  = os.environ.get("QWEN_API_KEY")  or os.environ.get("LLM_API_KEY") or "EMPTY"
    if base_url and OpenAI is not None:
        return OpenAI(base_url=base_url, api_key=api_key)
    if os.environ.get("OLLAMA_HOST") or shutil.which("ollama"):
        return None  # resolved lazily via ollama
    raise RuntimeError("No endpoint: set QWEN_BASE_URL/QWEN_API_KEY or run Ollama.")

_client = build_client()
print("model:", MODEL, "| client:", "openai" if _client else "ollama")

## 3. Prompt template

Requests a single declarative JSON (the same schema the cultural reference
models are authored in) with no prose, so the strict **Lane 0** parse succeeds.

In [ ]:
SYSTEM = "You are a world-modeling assistant. Return ONLY valid JSON. No markdown fences or commentary."

USER_TMPL = """Model this situation as a finite POSG-style world model. Return EXACTLY ONE JSON object:
{
  "players": [ {"id":"...", "description":"..."} ],
  "initial_state": "...",
  "states": { "<id>": {"description":"...", "terminal": false} },
  "actions": { "<state>": {"<player_id>": ["action1", "action2"]} },
  "transition_probabilities": { "<state>": {"action_or_joint": {"<next_state>": 1.0}} }
}
Include every stakeholder as a player and all reasonable alternative actions.
Do not invent facts the situation leaves open. No markdown fences.

SITUATION:
{case_input}

EXPECTED COVERAGE:
{judging}
"""


def build_prompt(case) -> dict:
    judging = "\n".join("EXPECT: " + (e.raw or "") for e in case.expected)
    judging += "\n" + "\n".join("AVOID: " + (f.raw or "") for f in case.forbidden)
    return {"system": SYSTEM,
            "user": USER_TMPL.format(case_input=case.llm_input, judging=judging)}


def _complete(prompt, temperature=0.4, max_tokens=4096):
    if _client is not None:
        resp = _client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": prompt["system"]},
                      {"role": "user", "content": prompt["user"]}],
            temperature=temperature, max_tokens=max_tokens,
        )
        return resp.choices[0].message.content
    import ollama
    resp = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "system", "content": prompt["system"]},
                  {"role": "user", "content": prompt["user"]}],
        options={"temperature": temperature, "num_predict": max_tokens},
    )
    return resp["message"]["content"]

## 4. Generate + evaluate every case (both banks)

For each of C1-C14 one candidate is generated from the **with-location** case and
one from the **without-location** case, run through the full three-lane pipeline,
and stashed under `experiments/cultural_qwen/`.

In [ ]:
def strip_code_fences(text: str) -> str:
    t = text.strip()
    if t.startswith("```"):
        t = t.split("\n", 1)[1]
        if t.rstrip().endswith("```"):
            t = t.rstrip()[:-3]
    return t.strip()

replies = {"with_loc": {}, "without_loc": {}}
for tag in replies:
    p = OUT_RUN / f"replies_{tag}.json"
    if p.exists():
        replies[tag] = json.loads(p.read_text(encoding="utf-8"))

records_with_loc, records_without = [], []
for cid in ORDER:
    for tag, bank, records in (("with_loc", bank_with_loc, records_with_loc),
                               ("without_loc", bank_without, records_without)):
        case = bank[cid]
        if cid not in replies[tag]:
            replies[tag][cid] = _complete(build_prompt(case))
            time.sleep(0.2)
        raw = replies[tag][cid]
        rec = evaluate_candidate(case, strip_code_fences(raw),
                                 {"model": "qwen-vl-72b", "sample": tag})
        rec["_raw"] = raw
        records.append(rec)
        print(cid, tag, "schema=", rec["schema"], "lane1=",
              (rec.get("lane1") or {}).get("verdict"))

for tag in replies:
    (OUT_RUN / f"replies_{tag}.json").write_text(
        json.dumps(replies[tag], indent=2), encoding="utf-8")
n_pass_with = sum(1 for r in records_with_loc
                  if r["lane0"] == "PASS" and (r.get("lane1") or {}).get("verdict") == "PASS")
n_pass_without = sum(1 for r in records_without
                     if r["lane0"] == "PASS" and (r.get("lane1") or {}).get("verdict") == "PASS")
print(f"PASS Lane0+1 -> with-loc {n_pass_with}/14  without-loc {n_pass_without}/14")

## 5. Lane results table

In [ ]:
import pandas as pd

def row(rec):
    l1 = rec["lane1"] or {}
    l3 = rec["lane3"] or {}
    return {
        "case": rec["case"], "schema": rec["schema"] or "-",
        "lane0": rec["lane0"], "lane1": l1.get("verdict") or "-",
        "fails": l1.get("n_fail", "-"), "warns": l1.get("n_warn", "-"),
        "coverage": round((l3.get("coverage") or 0.0), 3),
        "avoidance": round((l3.get("avoidance") or 0.0), 3),
        "lane3": round((l3.get("lane3") or 0.0), 3),
    }

df_with = pd.DataFrame([row(r) for r in records_with_loc])
df_without = pd.DataFrame([row(r) for r in records_without])
pd.set_option("display.width", 180)
print("==== WITH location ====")
print(df_with.to_string(index=False))
print()
print("==== WITHOUT location ====")
print(df_without.to_string(index=False))

## 6. Same-level expected-output check (with-loc vs without-loc)

Compares each case's lane3 (and structural counts) across the two banks. A small
delta is expected; a large drop flags a case where the model leaned on the
country label.

In [ ]:
def best_l3(records):
    by = {}
    for r in records:
        by.setdefault(r["case"], []).append((r["lane3"] or {}).get("lane3") or 0.0)
    return {c: float(np.mean(v)) for c, v in by.items()}

l3_with = best_l3(records_with_loc)
l3_without = best_l3(records_without)
print(f"{'Case':5} {'with_loc':>10} {'without':>10} {'delta':>8}")
summary_rows = []
for cid in ORDER:
    a, b = l3_with.get(cid, 0.0), l3_without.get(cid, 0.0)
    print(f"{cid:5} {a:10.3f} {b:10.3f} {b - a:8.3f}")
    summary_rows.append({"case": cid, "with_loc": a, "without_loc": b, "delta": b - a})
summary = pd.DataFrame(summary_rows)
print()
print(f"mean lane3  with_loc={summary['with_loc'].mean():.3f}  "
      f"without_loc={summary['without_loc'].mean():.3f}")
big_drops = summary[summary["delta"] < -0.1]
print("cases with >0.10 drop:", list(big_drops.case) if len(big_drops) else "none")

## 7. Structural EDA + heatmaps

For every candidate that passed Lane 1 we run `WorldModelEDA` and render two
heatmap families: a **cross-case metric heatmap** and **per-case transition
probability heatmaps** (source x target), exactly as in the reference EDA report.

In [ ]:
HEAT_STYLE = "YlGnBu"

def with_eda(records):
    return [r for r in records if r.get("eda")]

def save_cross_case_heatmap(records, path, title):
    data = with_eda(records)
    if len(data) < 2:
        print("skip heatmap: too few cases with EDA"); return
    cases = [r["case"] for r in data]
    metrics = ["states", "terminal", "joint_actions", "mean_branch",
               "mean_entropy", "max_depth", "sccs"]
    M = np.zeros((len(cases), len(metrics)))
    for i, r in enumerate(data):
        e = r["eda"]
        M[i, 0] = e["states"]["state_count"]
        M[i, 1] = e["states"]["terminal_state_count"]
        M[i, 2] = e["actions"]["joint_action_count"]
        M[i, 3] = e["states"]["mean_branching_factor"] or 0
        M[i, 4] = e["transitions"]["mean_transition_entropy"] or 0
        M[i, 5] = e["states"]["maximum_trajectory_depth"] or 0
        M[i, 6] = e["graph"]["strongly_connected_component_count"]
    N = np.zeros_like(M)
    for c in range(M.shape[1]):
        lo, hi = M[:, c].min(), M[:, c].max()
        if hi > lo:
            N[:, c] = (M[:, c] - lo) / (hi - lo)
    fig, ax = plt.subplots(figsize=(10, 6.0))
    ax.imshow(N, cmap=HEAT_STYLE, aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(len(metrics)), [m.replace("_", " ") for m in metrics],
                  rotation=30, ha="right")
    ax.set_yticks(range(len(cases)), cases)
    for i in range(len(cases)):
        for j in range(len(metrics)):
            ax.text(j, i, f"{M[i, j]:g}", ha="center", va="center", fontsize=8,
                    color="#0a1628" if 0.35 < N[i, j] < 0.75 else "white")
    ax.set_title(title)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("wrote", path)

save_cross_case_heatmap(records_with_loc, OUT_RUN / "compare_with_loc.png",
                        "Cultural candidates WITH location - metric heatmap")
save_cross_case_heatmap(records_without, OUT_RUN / "compare_without_loc.png",
                        "Cultural candidates WITHOUT location - metric heatmap")

In [ ]:
def save_transition_heatmap(records, path, title):
    data = with_eda(records)
    if not data:
        return
    cols, rows = 3, (len(data) + 2) // 3
    fig, axes = plt.subplots(max(rows, 1), cols,
                             figsize=(4.4 * cols, 3.2 * max(rows, 1)),
                             constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()
    last = None
    for i, r in enumerate(data):
        ax = axes[i]
        d = r["declarative"]
        states = list(d["states"])
        T = np.zeros((len(states), len(states)))
        for src, outs in (d.get("transition_probabilities") or {}).items():
            if src not in states:
                continue
            for nxt, p in outs.items():
                if nxt in states:
                    T[states.index(src), states.index(nxt)] += float(p)
        last = ax.imshow(T, cmap="inferno", vmin=0, vmax=1)
        ax.set_xticks(range(len(states)), states, rotation=45, ha="right", fontsize=6)
        ax.set_yticks(range(len(states)), states, fontsize=6)
        ax.set_title(r["case"], fontsize=9)
    for i in range(len(data), len(axes)):
        axes[i].axis("off")
    fig.suptitle(title, fontweight="bold", y=1.0)
    if last is not None:
        fig.colorbar(last, ax=axes, shrink=0.6, label="probability")
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("wrote", path)

save_transition_heatmap(records_with_loc, OUT_RUN / "transitions_with_loc.png",
                        "Cultural candidates WITH location - transition heatmaps")
save_transition_heatmap(records_without, OUT_RUN / "transitions_without_loc.png",
                        "Cultural candidates WITHOUT location - transition heatmaps")

## 8. Persist full results

In [ ]:
def safe(rec):
    return {k: v for k, v in rec.items()
            if not k.startswith("_")
            and isinstance(v, (str, int, float, bool, type(None), list, dict))}

payload = {
    "model": MODEL,
    "backend": ("openai" if _client is not None else "ollama"),
    "with_loc": [{**safe(r), "_raw": r.get("_raw")} for r in records_with_loc],
    "without_loc": [{**safe(r), "_raw": r.get("_raw")} for r in records_without],
}
out_path = OUT_RUN / "cultural_qwen_results.json"
out_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("wrote", out_path)
print("done: all", len(ORDER), "cases evaluated in both banks.")